# K3b — Cross-Encoder LoRA Fine-Tune + Eval Gate (spec 53_K3b)

LoRA fine-tunes `BAAI/bge-reranker-v2-m3` on denoised hard negatives sampled from the fused
candidate pool, using a grouped listwise-softmax loss weighted by goal-progress labels.
The adapter is then evaluated as a final-stage reranker stacked on top of K2 (`ChainReranker(k2, k3)`).
OOF scores from the fine-tuned cross-encoder are injected back into K2 as a `ce_ft_score` feature.

**Architecture:** K2 (LGBM LambdaMART) → K3b (bge-reranker-v2-m3 + LoRA adapter, top-K re-score).
**Run on:** Colab T4/G4 (16 GB). **Spec:** `.claude/documents/features/53_K3b_ce_lora_finetune.md`.

## 1. Drive + HF auth (Colab Secrets)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/recsys2026'
os.environ['HF_HOME'] = f'{DRIVE}/hf_cache'
OUT = f'{DRIVE}/outputs'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
os.makedirs(OUT, exist_ok=True)
try:
    t = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = os.environ['HUGGINGFACE_HUB_TOKEN'] = t
    from huggingface_hub import login
    login(t)
    print('HF ok')
except Exception as e:
    print('no HF_TOKEN secret:', e)

## 2. Clone + install

In [ ]:
!git clone --branch fresh-start --depth 1 https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
# peft and trackio are added for K3b (LoRA adapter + loss logging)
!pip -q install datasets bm25s scipy scikit-learn lightgbm sentence-transformers numpy pandas peft trackio
import sys
sys.path.insert(0, '.')

## 3. Config

T4/G4 runtime levers are surfaced here with prominent comments. Tune these first when adjusting
the memory/speed/quality trade-off before touching deeper parameters.

In [ ]:
# ── Cross-encoder model + retrieval budget ──────────────────────────────────
CE_MODEL = 'BAAI/bge-reranker-v2-m3'  # bge-reranker-v2-m3: 568M XLM-R backbone
CROSS_ENCODER_K = 100   # candidates scored per turn at both train and serve
N_NEG = 15              # hard negatives sampled per gold (contrast knob)
K_MIN = 4               # minimum negatives required to keep a training turn
FOLDS = 3               # k-folds for OOF stacking (FOLDS+1 total fine-tunes)
SEED = 0

# ── Sequence / dtype (train==serve — do NOT split these) ─────────────────────
MAX_LEN = 2048          # ← T4/G4: must match at serve; lower to 1536 if OOM after A1 re-measure
MAX_DOC_TOK = 1100      # doc-side token cap (query preserved); pair = query + doc + 4 specials
DTYPE = 'auto'          # 'auto' → fp16 on T4 (no bf16), bf16 on Ampere+; 'fp16'/'bf16' to force

# ── Training-data subset ─────────────────────────────────────────────────────
# Raise on a faster GPU (A100/H100) for full coverage; lower to 1000–2000 if OOM or time-budget tight.
TRAIN_SUBSET = 4000     # max train SESSIONS retained for the single fine-tune (not OOF)

# ── Val nDCG@20 callback subset ──────────────────────────────────────────────
# Small fixed dev subset for the real nDCG@20 early-stopping callback (spec §4.5/§4.7).
# Bounds per-epoch eval cost: ~300 turns ≈ 1-2 min/epoch on T4 at CROSS_ENCODER_K=100.
# Raise (e.g. to 500) on A100 for tighter signal; lower (e.g. to 150) if per-epoch eval is slow.
VAL_NDCG_TURNS = 300

# ── LoRA config ──────────────────────────────────────────────────────────────
LORA = {
    'r': 16,                          # rank: 8 = memory-light, 16 = standard (our default)
    'alpha': 32,                      # scaling = alpha/r = 2 (common default)
    'dropout': 0.05,
    'target_modules': ['query', 'value'],   # XLM-R attention projections
}

# ── Training loop ─────────────────────────────────────────────────────────────
# T4/G4 tuning: if OOM lower batch_groups (→1) or MAX_LEN (→1536 after A1 re-measure).
# Cost ≈ FOLDS+1 fine-tunes total (1 for the gate model + FOLDS for OOF stacking).
# Effective batch = batch_groups * grad_accum = 2 * 16 = 32 groups (stability without OOM at seq=2048).
TRAIN = {
    'epochs': 3,                   # upper bound; early-stop (patience=1) picks the real stop
    'lr': 1e-4,
    'weight_decay': 0.0,
    'batch_groups': 2,             # ← micro-batch size; lower first if OOM
    'grad_accum': 16,              # gradient-accumulation steps → effective batch = 2*16 = 32
    'warmup': 0.05,                # fraction of total optimizer steps for linear warmup
    'log_every': 50,               # log to Trackio every N optimizer steps
    'group_by_length': True,       # length-grouped sampler (less pad waste on T4)
    'early_stop_patience': 1,      # stop after N epochs without val nDCG@20 improvement
}

# ── Retrieval + dense model (shared with K2 setup) ───────────────────────────
TOPK = 500
DENSE_MODEL = 'BAAI/bge-large-en-v1.5'
DENSE_QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
CONTENT_MODALITIES = {
    'cknn_audio': 'audio-laion_clap',
    'cknn_attr': 'attributes-qwen3_embedding_0.6b',
}
ORG = 'talkpl-ai'
import glob
ENRICHED_GLOB = f'{OUT}/catalog_enriched_*.parquet'

print('Config OK — CE_MODEL:', CE_MODEL, '| MAX_LEN:', MAX_LEN, '| DTYPE:', DTYPE,
      '| VAL_NDCG_TURNS:', VAL_NDCG_TURNS)

## 4. Load data + catalog (enriched) + channels + RRFFusion + train K2

Reuses the phase2_rerank setup: Conversations, Catalog from enriched parquet, channels, fusion,
FeatureBuilder + dense_cos, LGBMReranker.fit. K3b stacks on top of K2 so K2 must be trained first.
Asserts 100% enriched coverage over the candidate pool (spec §2).

In [ ]:
import os, glob, pickle
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

from mcrs.data.catalog import Catalog
from mcrs.data.embeddings import TrackEmbeddings, UserEmbeddings
from mcrs.data.conversations import Conversations
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.retrieval.dense_channel import DenseChannel
from mcrs.retrieval.personalization import ContentKNNChannel, CFChannel, SameArtistChannel
from mcrs.retrieval.related_artist import RelatedArtistChannel, build_artist_cooc, tid_to_artists_from_catalog
from mcrs.retrieval.fusion import RRFFusion
from mcrs.rerank.features import FeatureBuilder
from mcrs.rerank.lgbm import LGBMReranker
from mcrs.rerank.train import build_rerank_groups

# ── Catalog (enriched from A1 parquet) ──────────────────────────────────────
meta_rows = load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')
_enr_files = sorted(glob.glob(ENRICHED_GLOB))
assert _enr_files, f'No enriched parquet found at {ENRICHED_GLOB} — run A1 notebook first'
_edf = pd.read_parquet(_enr_files[-1])
enr = dict(zip(_edf['track_id'], _edf['enriched_doc']))
cat = Catalog(meta_rows, enriched_docs=enr)
USE_ENRICHED = True
print(f'enriched docs: {len(enr)} (from {_enr_files[-1].split("/")[-1]})')

# ── Spec §2: assert 100% enriched coverage (hard fail — K3b needs enriched docs everywhere) ──
_sample_pool_tids = list(cat._meta.keys())[:500]
assert all(cat.is_enriched(t) for t in _sample_pool_tids), \
    'K3b requires 100% enriched doc coverage over the catalog (spec §2)'
print('enriched coverage check: OK')

# ── Track/User embeddings ──────────────────────────────────────────────────
tre = load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Embeddings', split='all_tracks')
_avail = set(tre.column_names)
CKNN_MODS = {lab: mod for lab, mod in CONTENT_MODALITIES.items() if mod in _avail}
te = {lab: TrackEmbeddings(tre.select_columns(['track_id', mod]), modalities=[mod])
      for lab, mod in CKNN_MODS.items()}
te_cf = TrackEmbeddings(tre.select_columns(['track_id', 'cf-bpr']), modalities=['cf-bpr'])
ued = load_dataset(f'{ORG}/TalkPlayData-Challenge-User-Embeddings')
ue = UserEmbeddings([r for sp in ued for r in ued[sp]])

# ── Conversations ──────────────────────────────────────────────────────────
dsd = load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset')
_n_tr = min(TRAIN_SUBSET, len(dsd['train']))
conv_tr = Conversations(dsd['train'].select(range(_n_tr)))
conv_dv = Conversations(dsd['test'])
print(f'sessions: train={_n_tr}, dev={len(dsd["test"])}')

# ── Dense model + doc matrix ──────────────────────────────────────────────
model = SentenceTransformer(DENSE_MODEL, device='cuda')
doc_mat = model.encode(
    [cat.id_to_metadata(t, enriched=USE_ENRICHED) for t in cat.index_to_id],
    batch_size=256, normalize_embeddings=True, show_progress_bar=True,
)

# ── Related-artist co-occurrence ──────────────────────────────────────────
COOC_PKL = f'{OUT}/artist_cooc.pkl'
if os.path.exists(COOC_PKL):
    cooc = pickle.load(open(COOC_PKL, 'rb'))
    print('loaded cooc', len(cooc), 'artists')
else:
    cooc = build_artist_cooc(dsd['train'], tid_to_artists_from_catalog(cat))
    pickle.dump(cooc, open(COOC_PKL, 'wb'))
    print('built cooc', len(cooc), 'artists')

# ── Channels + fusion ────────────────────────────────────────────────────
dense = DenseChannel(
    cat.index_to_id, doc_mat,
    lambda qs: model.encode([DENSE_QUERY_PREFIX + q for q in qs],
                             batch_size=256, normalize_embeddings=True),
    normalize=False,
)
cknn = [ContentKNNChannel(te[lab], mod, label=lab) for lab, mod in CKNN_MODS.items()]
chans = [
    BM25Channel(cat, enriched=USE_ENRICHED),
    dense,
    *cknn,
    CFChannel(ue, te_cf, 'cf-bpr'),
    SameArtistChannel(cat),
    RelatedArtistChannel(cat, cooc),
]
fusion = RRFFusion(chans, k=60)
labels = [c.label for c in chans]
print('channels:', labels)

# ── K2 — FeatureBuilder + dense_cos + LGBMReranker ───────────────────────
# K2 uses the plain (non-enriched) QueryBuilder for retrieval; K3b uses the enriched one below.
qb_plain = QueryBuilder()
_qcache = {}

def precompute_qvecs(turns):
    texts = [DENSE_QUERY_PREFIX + qb_plain.build(t).text for t in turns]
    mat = model.encode(texts, batch_size=256, normalize_embeddings=True)
    _qcache.update({(t.session_id, t.turn_number): mat[i] for i, t in enumerate(turns)})

def dense_cos(ctx, tid):
    j = cat.id_to_index.get(tid)
    if j is None:
        return 0.0
    v = _qcache.get((ctx.session_id, ctx.turn_number))
    if v is None:
        v = model.encode([DENSE_QUERY_PREFIX + qb_plain.build(ctx).text],
                         normalize_embeddings=True)[0]
        _qcache[(ctx.session_id, ctx.turn_number)] = v
    return float(v @ doc_mat[j])

fb = FeatureBuilder(cat, labels, score_fns={'dense_cos': dense_cos})

tr = list(conv_tr.turns())
precompute_qvecs(tr)
print(f'train turns: {len(tr)} — building K2 groups (runs fusion + dense over train)...')
k2_groups = build_rerank_groups(
    qb_plain, fusion, tr,
    lambda t: conv_tr.gold(t.session_id, t.turn_number),
    topk=TOPK,
)
k2 = LGBMReranker(
    fb, n_estimators=500, neg_cap=150, early_stopping_rounds=50, val_fraction=0.1,
).fit(k2_groups)
print(f'K2 fit: train_groups={k2.n_train_groups_} val_groups={k2.n_val_groups_}'
      f' best_iter={getattr(k2.model, "best_iteration_", None)}')
k2.save(f'{OUT}/k2_lgbm.txt')
print('K2 saved ->', f'{OUT}/k2_lgbm.txt')

## 5. Build enriched QueryBuilder + CE training groups

Builds `(query_text, [pos_doc, neg_doc...], group_weight)` triples using:
- Enriched `QueryBuilder(markers=True, taste_items=5)` for richer query context
- `build_ce_training_groups` with denoised hard negatives + goal-progress weighting
- `goal_progress_assessments` from the raw session row

In [ ]:
from mcrs.retrieval.query import QueryBuilder
from mcrs.training.ce_data import build_ce_training_groups

# Enriched query builder: markers=True adds request:/context:/goal:/taste: sections;
# taste_items=5 appends the 5 most-recent liked tracks (newest-first) for warm sessions.
track_label = lambda tid: (
    f"{cat.metadata(tid).get('artist_name', '')} – {cat.metadata(tid).get('track_name', '')}"
    if tid in cat._meta else None
)
qb_ce = QueryBuilder(markers=True, taste_items=5, track_label_fn=track_label)

# goal_progress_assessments is a list of {turn_number, goal_progress_assessment} per session row.
# Build a lookup per session_id so gp_fn(turn) -> label | None is O(1).
_gp_lookup = {}
for sid, row in conv_tr._rows.items():
    _gp_lookup[sid] = {
        int(a['turn_number']): a['goal_progress_assessment']
        for a in (row.get('goal_progress_assessments') or [])
    }

def gp_fn(turn):
    """Return the goal_progress_assessment label for this turn, or None if missing."""
    return _gp_lookup.get(turn.session_id, {}).get(turn.turn_number)

report = {}
groups = build_ce_training_groups(
    qb_ce, fusion, tr,
    lambda t: conv_tr.gold(t.session_id, t.turn_number),
    catalog=cat,
    cross_encoder_k=CROSS_ENCODER_K,
    n_negatives=N_NEG,
    k_min=K_MIN,
    seed=SEED,
    gp_fn=gp_fn,
    report=report,
)
print('build_ce_training_groups report:', report)
# report keys: dropped_no_gold (gold not in top-K pool), dropped_few_neg (<k_min negs), kept
kept_turns = tr  # NOTE: groups is aligned to tr (same length after drops are recorded in report)
print(f'groups built: {len(groups)} | example query[:80]: {groups[0][0][:80] if groups else "(empty)"}')

## 6. Session-disjoint train/val split + Trackio init + LoRA fine-tune

Uses fold 0 as the validation set (10% of sessions). `val_eval_fn=make_val_ndcg` wires the
real nDCG@20 ranking metric as the early-stopping signal (spec §4.5/§4.7): a closure over the
IN-TRAINING model scores a fixed `dev[:VAL_NDCG_TURNS]` subset per epoch via ChainReranker(k2, k3_tmp),
bounded to `VAL_NDCG_TURNS` turns to cap per-epoch cost. The best-checkpoint is the epoch with the
highest val nDCG@20, not the lowest val loss.

The trained adapter is pushed to HF Hub by revision for provenance.

In [ ]:
import os
import torch
import trackio
from mcrs.training.ce_data import assign_session_folds
from mcrs.training.ce_finetune import finetune_cross_encoder
from mcrs.rerank.cross_encoder import doc_token_budget, truncate_doc_tokens
from mcrs.rerank.neural import NeuralReranker, ChainReranker
from mcrs.filter.assembly import TopKAssembler
from mcrs.run.harness import InferenceHarness
from mcrs.eval.harness import GoldRow
from mcrs.eval.official import score_official

# ── Val subset size for real nDCG@20 callback (spec §4.5/§4.7) ──────────────
# Bounded to VAL_NDCG_TURNS to cap per-epoch cost; ~300 turns ≈ 1-2 min/epoch on T4 at K=100.
VAL_NDCG_TURNS = 300

# Session-disjoint 10-fold split: fold 0 = validation (~10% of sessions); rest = train.
# assign_session_folds guarantees no session straddles two folds (leak-safe).
folds_all = assign_session_folds([t.session_id for t in kept_turns], k=10, seed=SEED)

# Rebuild per-group fold assignment aligned to kept_turns (same order as groups).
_gold_fn = lambda t: conv_tr.gold(t.session_id, t.turn_number)
_pool_ids_sets = [set(c.track_id for c in fusion.fuse(
    [qb_ce.build(t).text], CROSS_ENCODER_K,
    topk_internal=CROSS_ENCODER_K,
    batch_context=[{'history_tids': t.history_tids, 'user_id': t.user_id}],
    user_ids=[t.user_id],
)[0][:CROSS_ENCODER_K]) for t in kept_turns]
_gold_in_pool_flags = [
    (g is not None) and (g in pids)
    for t, pids in zip(kept_turns, _pool_ids_sets)
    for g in [_gold_fn(t)]
]
_group_folds = [f for t, f, ok in zip(kept_turns, folds_all, _gold_in_pool_flags) if ok]

if len(_group_folds) != len(groups):
    _sid_to_fold = dict(zip([t.session_id for t in kept_turns], folds_all))
    _kept_sids_tmp = [t.session_id for t, ok in zip(kept_turns, _gold_in_pool_flags) if ok]
    _group_folds = [_sid_to_fold[sid] for sid in _kept_sids_tmp]

assert len(_group_folds) == len(groups), \
    f'fold/group mismatch: {len(_group_folds)} folds vs {len(groups)} groups'

tr_groups = [g for g, f in zip(groups, _group_folds) if f != 0]
va_groups = [g for g, f in zip(groups, _group_folds) if f == 0]
print(f'train groups: {len(tr_groups)} | val groups: {len(va_groups)}')

# ── Dev gold rows for the val nDCG@20 callback ───────────────────────────────
dv = list(conv_dv.turns())
precompute_qvecs(dv)
_dv_subset = dv[:VAL_NDCG_TURNS]
_golds_subset = [
    GoldRow(t.session_id, t.user_id, t.turn_number,
            conv_dv.gold(t.session_id, t.turn_number))
    for t in _dv_subset
]
_asm_val = TopKAssembler(cat)

# ── val nDCG@20 closure wired to the IN-TRAINING model ───────────────────────
# Spec §4.5/§4.7: real ranking metric drives best-checkpoint + early stopping.
# Uses the live model + tok (NOT build_cross_encoder_score_fn, which loads a fresh CrossEncoder).
# Doc truncation via doc_token_budget/truncate_doc_tokens matches serve exactly (train==serve).
def make_val_ndcg(model, tok):
    """Return val nDCG@20 over _dv_subset using the in-training model (real ranking metric, spec §4.5/§4.7).
    Bounded to VAL_NDCG_TURNS for cost (~1-2 min/epoch on T4 at CROSS_ENCODER_K=100)."""
    dev = next(model.parameters()).device
    use_amp = (str(dev).startswith('cuda'))
    try:
        amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
    except Exception:
        amp_dtype = torch.float16

    enc = lambda s: tok.encode(s, add_special_tokens=False, truncation=True, max_length=MAX_LEN)

    def _score_fn(pairs):
        queries = [q for q, _ in pairs]
        # apply per-pair doc token budget (same as serve: preserve query, silence 512>512 warning)
        docs = []
        for q, d in pairs:
            budget = doc_token_budget(len(enc(q)), MAX_LEN, MAX_DOC_TOK)
            docs.append(truncate_doc_tokens(enc, tok.decode, d, budget))
        feats = tok(queries, docs, padding=True, truncation=True,
                    max_length=MAX_LEN, return_tensors='pt')
        feats = {k: v.to(dev) for k, v in feats.items()}
        with torch.inference_mode():
            with torch.autocast(device_type=str(dev).split(':')[0], dtype=amp_dtype, enabled=use_amp):
                scores = model(**feats).logits.squeeze(-1)
        return scores.float().cpu().tolist()

    # wrap as NeuralReranker + ChainReranker(k2, k3_tmp) over the fixed dev subset
    k3_tmp = NeuralReranker(cat, qb_ce, _score_fn, cross_encoder_k=CROSS_ENCODER_K, enriched=True)
    rows = InferenceHarness(qb_plain, fusion, _asm_val,
                            reranker=ChainReranker(k2, k3_tmp), topk=TOPK).run(_dv_subset)
    return score_official(rows, _golds_subset, len(cat))['ndcg@20']

trackio.init(project='k3b-ce-lora')
CKPT_DIR = f'{OUT}/ckpt_k3b'
os.makedirs(CKPT_DIR, exist_ok=True)

# val_eval_fn=make_val_ndcg: real nDCG@20 over VAL_NDCG_TURNS dev turns drives early stopping
# and best-checkpoint selection (spec §4.5/§4.7). Falls back to -val_loss only if None.
adapter = finetune_cross_encoder(
    tr_groups, va_groups,
    base_model=CE_MODEL,
    lora_cfg=LORA,
    max_length=MAX_LEN,
    max_doc_tokens=MAX_DOC_TOK,
    dtype=DTYPE,
    train_cfg=TRAIN,
    logger=trackio,
    out_dir=CKPT_DIR,
    val_eval_fn=make_val_ndcg,
)
print('adapter saved to:', adapter)

# Push adapter to HF Hub by revision for provenance (spec §4.7).
if adapter:
    from huggingface_hub import HfApi
    _api = HfApi()
    HUB_ADAPTER_REPO = f'{ORG}/k3b-lora-adapter'
    try:
        _api.create_repo(HUB_ADAPTER_REPO, exist_ok=True, private=True)
        _api.upload_folder(
            folder_path=adapter,
            repo_id=HUB_ADAPTER_REPO,
            commit_message=f'k3b adapter TRAIN_SUBSET={TRAIN_SUBSET} MAX_LEN={MAX_LEN} LORA_R={LORA["r"]}',
        )
        print('pushed adapter to Hub:', HUB_ADAPTER_REPO)
    except Exception as e:
        print('Hub push skipped (ok in offline/test run):', e)

## 7. Eval gate: K2 vs K2+K3b on dev (overall + per-segment + per-goal-progress)

Ships only if `K2+K3ft > K2`. Also slices by segment (cold/warm) and by the gold turn's
goal-progress label. Abort the goal-progress lever if off-goal-gold nDCG regresses (spec §4.3).

In [ ]:
import json
from mcrs.rerank.cross_encoder import build_cross_encoder_score_fn
from mcrs.rerank.neural import NeuralReranker, ChainReranker
from mcrs.filter.assembly import TopKAssembler
from mcrs.run.harness import InferenceHarness, validate_submission
from mcrs.eval.harness import GoldRow
from mcrs.eval.official import score_official

# Load the fine-tuned cross-encoder score_fn (adapter merged into base for fast inference).
ce_score = build_cross_encoder_score_fn(
    CE_MODEL, device='cuda',
    max_length=MAX_LEN,
    max_doc_tokens=MAX_DOC_TOK,
    dtype=DTYPE,
    lora_adapter=adapter,   # local ckpt dir; or HUB_ADAPTER_REPO for Hub-loaded
)

# K3b reranker: uses the enriched QueryBuilder (same as training — train==serve)
k3b = NeuralReranker(
    cat, qb_ce, ce_score,
    cross_encoder_k=CROSS_ENCODER_K,
    enriched=True,
)

# Dev turns + gold rows
dv = list(conv_dv.turns())
precompute_qvecs(dv)   # batch encode dev queries into _qcache for dense_cos feature
golds = [
    GoldRow(t.session_id, t.user_id, t.turn_number,
            conv_dv.gold(t.session_id, t.turn_number))
    for t in dv
]
keys = [(g.session_id, g.turn_number) for g in golds]
asm = TopKAssembler(cat)

# Run K2 and K2+K3b inference
k2_rows = InferenceHarness(qb_plain, fusion, asm, reranker=k2, topk=TOPK).run(dv)
k3b_rows = InferenceHarness(qb_plain, fusion, asm, reranker=ChainReranker(k2, k3b), topk=TOPK).run(dv)
validate_submission(k2_rows, catalog=cat, expected_keys=keys)
validate_submission(k3b_rows, catalog=cat, expected_keys=keys)

s_k2 = score_official(k2_rows, golds, len(cat))
s_k3b = score_official(k3b_rows, golds, len(cat))
print(f'Overall nDCG@20:  K2={round(s_k2["ndcg@20"], 4)}  K2+K3b={round(s_k3b["ndcg@20"], 4)}')
delta = s_k3b['ndcg@20'] - s_k2['ndcg@20']
print(f'Delta: {round(delta, 4)} ({"SHIPS" if delta > 0 else "ABORT — no gain"})')

# Per-segment slice (cold/warm)
def _seg_score(rows, g_list, seg):
    gs = [g for g, t in zip(g_list, dv) if t.segment == seg]
    if not gs:
        return None
    ks = {(g.session_id, g.turn_number) for g in gs}
    rs = [r for r in rows if (r.session_id, r.turn_number) in ks]
    return round(score_official(rs, gs, len(cat))['ndcg@20'], 4)

for seg in ('cold', 'warm'):
    print(f'  {seg}: K2={_seg_score(k2_rows, golds, seg)}  K2+K3b={_seg_score(k3b_rows, golds, seg)}')

# Per-goal-progress label slice (from dev conversation rows)
_gp_dv = {}
for sid, row in conv_dv._rows.items():
    for a in (row.get('goal_progress_assessments') or []):
        _gp_dv[(sid, int(a['turn_number']))] = a['goal_progress_assessment']

def _gp_score(rows, g_list, label):
    gs = [g for g in g_list if _gp_dv.get((g.session_id, g.turn_number)) == label]
    if not gs:
        return None
    ks = {(g.session_id, g.turn_number) for g in gs}
    rs = [r for r in rows if (r.session_id, r.turn_number) in ks]
    return round(score_official(rs, gs, len(cat))['ndcg@20'], 4)

for lbl in ('MOVES_TOWARD_GOAL', 'DOES_NOT_MOVE_TOWARD_GOAL'):
    v_k2 = _gp_score(k2_rows, golds, lbl)
    v_k3b = _gp_score(k3b_rows, golds, lbl)
    print(f'  goal_progress={lbl}: K2={v_k2}  K2+K3b={v_k3b}')
    # Spec §4.3 guard: abort goal-progress lever if off-goal-gold nDCG regresses.
    if lbl == 'DOES_NOT_MOVE_TOWARD_GOAL' and v_k2 is not None and v_k3b is not None:
        if v_k3b < v_k2:
            print(f'  WARNING: off-goal-gold nDCG regressed {v_k3b} < {v_k2} — abort goal-progress lever (spec §4.3)')

# Persist results
_gate_result = {'k2': s_k2, 'k2_k3b': s_k3b, 'delta_ndcg20': delta,
                'ships': delta > 0, 'adapter': adapter}
json.dump(_gate_result, open(f'{OUT}/phase2_k3b_gate.json', 'w'), indent=2)
print('gate result saved ->', f'{OUT}/phase2_k3b_gate.json')
# Ships only if K2+K3b > K2; abort goal-progress lever if off-goal-gold nDCG regresses (spec §4.3)

## 8. OOF stacking: inject `ce_ft_score` as a K1 feature, retrain K2

Produces leak-free per-turn CE scores via `oof_ce_scores` (FOLDS-fold session-disjoint cross-fit).
Each fold is fine-tuned on the other folds and scores its held-out fold's candidates with
`normalize_within_pool` per turn. The OOF scores are injected as `ce_ft_score` into FeatureBuilder
and K2 is retrained.

**Cost:** FOLDS+1 fine-tunes total (FOLDS for OOF + 1 for the gate). Lower FOLDS to 2 if time-budget
is tight; the OOF scores are only meaningful if each fold is large enough (~300+ train groups).

In [ ]:
# ── OPTIONAL HEAVY STEP — OOF stacking: inject `ce_ft_score` as a K1 feature, retrain K2 ──────
# Cost = FOLDS+1 fine-tunes total (FOLDS for OOF cross-fit + 1 already done above for the gate).
# Budget levers (tune before running on T4):
#   FOLDS=2  → minimum for full coverage (each fold trained on 2/3 of turns; heavier than 3-fold)
#   fold epochs=1 or 2 (fold models only need leak-free scores, not the best serve model)
#   TRAIN_SUBSET, MAX_LEN — same as above
# NOTE: run this ONLY if the gate eval (cell 7) already showed K2+K3 > K2 (otherwise no point
# injecting OOF CE scores into K2 when the CE hasn't proven it adds ranking signal).

import os, json
import torch
from mcrs.training.ce_finetune import oof_ce_scores, finetune_cross_encoder
from mcrs.training.ce_data import normalize_within_pool
from mcrs.rerank.cross_encoder import build_cross_encoder_score_fn
from mcrs.rerank.features import FeatureBuilder
from mcrs.rerank.lgbm import LGBMReranker
from mcrs.filter.assembly import TopKAssembler
from mcrs.run.harness import InferenceHarness
from mcrs.eval.official import score_official

# ── Re-collect (session_id, turn_number) pairs that have a group (gold-in-pool, k_min ok) ──
_kept_sids = [t.session_id for t, ok in zip(kept_turns, _gold_in_pool_flags) if ok]
_kept_tnums = [t.turn_number for t, ok in zip(kept_turns, _gold_in_pool_flags) if ok]
oof_turn_keys = list(zip(_kept_sids, _kept_tnums))
assert len(oof_turn_keys) == len(groups), f'OOF key/group mismatch: {len(oof_turn_keys)} vs {len(groups)}'
print(f'OOF: {len(oof_turn_keys)} turns across {FOLDS} folds — cost = {FOLDS} fine-tunes')

# ── Group + query + pool lookup tables for fit_fn / score_fn ─────────────────
_group_by_key = {k: g for k, g in zip(oof_turn_keys, groups)}
_query_by_key = {(t.session_id, t.turn_number): qb_ce.build(t).text for t in kept_turns}

# Pre-fetch R7 fusion pools for all training turns (reuse qb_ce queries, CROSS_ENCODER_K depth).
print('Pre-fetching R7 fusion pools for all OOF training turns...')
_oof_queries = [_query_by_key[(t.session_id, t.turn_number)] for t in kept_turns]
_oof_bc = [{'history_tids': t.history_tids, 'user_id': t.user_id} for t in kept_turns]
_oof_uids = [t.user_id for t in kept_turns]
_oof_all_pools = fusion.fuse(
    _oof_queries, CROSS_ENCODER_K,
    topk_internal=CROSS_ENCODER_K,
    batch_context=_oof_bc,
    user_ids=_oof_uids,
)
_pool_by_key = {(t.session_id, t.turn_number): pool
                for t, pool in zip(kept_turns, _oof_all_pools)}

# ── fit_fn: train one fold's model, return its adapter path ──────────────────
# Fold models only need leak-free scores, not the best serve model → cheaper config:
# epochs=1 (or 2), same LORA, smaller batch if needed.
_FOLD_TRAIN_CFG = dict(TRAIN, epochs=1)   # fold models: 1 epoch is enough for leak-free scores

def fit_fn(train_rows, fold=0):
    """Fit a fold model on train_rows, return the adapter dir. Called by oof_ce_scores."""
    # train_rows: list of (session_id, turn_number, fold_id) for the training folds
    fold_groups = [_group_by_key[(sid, tn)] for sid, tn, _f in train_rows
                   if (sid, tn) in _group_by_key]
    # simple 10% val split within the fold-train set (not session-disjoint here — OOF
    # leak-safety is guaranteed at the fold level by oof_ce_scores, not by this inner split)
    _n_val = max(1, len(fold_groups) // 10)
    fold_tr = fold_groups[_n_val:]
    fold_va = fold_groups[:_n_val]
    _ckpt = f'{OUT}/ckpt_k3b_oof_fold{fold}'
    os.makedirs(_ckpt, exist_ok=True)
    _adapter = finetune_cross_encoder(
        fold_tr, fold_va,
        base_model=CE_MODEL,
        lora_cfg=LORA,
        max_length=MAX_LEN,
        max_doc_tokens=MAX_DOC_TOK,
        dtype=DTYPE,
        train_cfg=_FOLD_TRAIN_CFG,
        logger=trackio,
        out_dir=_ckpt,
        val_eval_fn=None,   # fold models: -val_loss is fine; no full nDCG eval per fold
    )
    return _adapter

# ── score_fn: score each candidate in the turn's pool, return {tid: score} ───
# Uses build_cross_encoder_score_fn (loads the merged adapter for fast inference),
# then normalizes within the pool (min-max, fold-scale invariant).
_loaded_score_fns = {}   # cache so each fold adapter is loaded only once

def score_fn(model_adapter, row):
    """Score the held-out turn's CROSS_ENCODER_K candidates. Returns {track_id: score}."""
    if model_adapter not in _loaded_score_fns:
        _loaded_score_fns[model_adapter] = build_cross_encoder_score_fn(
            CE_MODEL, device='cuda',
            max_length=MAX_LEN, max_doc_tokens=MAX_DOC_TOK, dtype=DTYPE,
            lora_adapter=model_adapter,
        )
    _sfn = _loaded_score_fns[model_adapter]
    key = (row['session_id'], row['turn'])
    pool = _pool_by_key.get(key, [])
    if not pool:
        return {}
    q = _query_by_key.get(key, '')
    docs = [cat.id_to_metadata(c.track_id, enriched=True) for c in pool]
    raw_scores = _sfn([(q, d) for d in docs])
    tid_to_raw = {c.track_id: s for c, s in zip(pool, raw_scores)}
    return normalize_within_pool(tid_to_raw)   # {track_id: normalized_score}

# ── Run OOF cross-fit ─────────────────────────────────────────────────────────
print('Running OOF cross-fit (FOLDS fine-tunes — heaviest cell)...')
oof = oof_ce_scores(
    oof_turn_keys, folds=FOLDS, seed=SEED,
    fit_fn=fit_fn,
    score_fn=score_fn,
)
# oof: {(session_id, turn_number, track_id): normalized_ce_score}
print(f'OOF scores collected: {len(oof)} (session, turn, track_id) triples')

# ── Inject ce_ft_score as a K1 feature + retrain K2 ─────────────────────────
def ce_ft_score(ctx, tid):
    """OOF-normalized CE score for (session_id, turn_number, track_id); 0.0 if absent."""
    return oof.get((ctx.session_id, ctx.turn_number, tid), 0.0)

fb_oof = FeatureBuilder(cat, labels, score_fns={'dense_cos': dense_cos, 'ce_ft_score': ce_ft_score})
print('Retraining K2 with ce_ft_score feature...')
k2_oof = LGBMReranker(
    fb_oof, n_estimators=500, neg_cap=150, early_stopping_rounds=50, val_fraction=0.1,
).fit(k2_groups)
print(f'K2+oof fit: train={k2_oof.n_train_groups_} val={k2_oof.n_val_groups_}')

# ── Eval K2-oof vs K2 (and vs K2+K3b) ────────────────────────────────────────
k2oof_rows = InferenceHarness(qb_plain, fusion, TopKAssembler(cat), reranker=k2_oof, topk=TOPK).run(dv)
s_k2oof = score_official(k2oof_rows, golds, len(cat))
print(f'K2 nDCG@20={round(s_k2["ndcg@20"],4)}  K2+oof={round(s_k2oof["ndcg@20"],4)}  K2+K3b={round(s_k3b["ndcg@20"],4)}')
json.dump({'k2': s_k2, 'k2_oof': s_k2oof, 'k2_k3b': s_k3b},
          open(f'{OUT}/phase2_k3b_oof.json', 'w'), indent=2)
print('OOF scores saved ->', f'{OUT}/phase2_k3b_oof.json')

## 9. Gate decision + next steps

**Ship criteria (spec §6):**
- `K2+K3b nDCG@20 > K2 nDCG@20` overall — otherwise revert to K2 alone.
- No regression on the `DOES_NOT_MOVE_TOWARD_GOAL` segment (§4.3 guard).
- `K2+oof nDCG@20 > K2 nDCG@20` — OOF ce_ft_score adds signal to LGBM features.

**If the gate passes:**
1. Retrain the full pipeline on train+dev (blind run).
2. The adapter on HF Hub is the artifact to package in the CodaBench submission.
3. Next lever: larger `TRAIN_SUBSET` (full train) or higher `CROSS_ENCODER_K` on an A100.

**If the gate fails:**
- K3b gain is bounded by recall@pool ≈ 0.65 — if the pool quality is low, fix R7 first.
- Try reducing `N_NEG` (15→8) for harder contrast, or `CROSS_ENCODER_K` (100→50) to reduce noise.
- Check whether cold turns (0 history) dominate the regression — they have no taste: clause.